# Day 4 — Security, configuration and message recovery (Jupyter + CMD)

**Prerequisites:** Day 1 setup is complete (`start-lab.bat`). The incident work from Days 2 and 3 gives useful context but is not required.

Today covers the three questions that come up whenever a Kafka ticket turns into a change request:

| Question | What you work with |
|----------|--------------------|
| **Who is allowed to do this?** | SCRAM authentication, ACLs, and the IAM listener |
| **What does the topic actually keep?** | Topic configuration and retention |
| **Can we get the messages back?** | Consumer offset reset and replay |

The third one is where the risk lives. Resetting offsets is one of the few Kafka operations that can cause real business damage — duplicate payments, duplicate emails, duplicate shipments — and it is often requested casually by someone who has not thought that through. A large part of today is learning to slow that request down and check it properly.

Every code cell runs CMD commands, the same ones listed in [commands.md](commands.md). Theory: [notes.md](notes.md).

**Why you stay on your own names.** Several seats share this cluster, exactly as several teams share brokers in production. A consumer group is that tea’ bookmark, so moving it affects their application and not yours. Use only your `%GROUP%` and `%TOPIC%` for the reset and replay work, and keep the permission exercise on `%ACL_TOPIC%` so a security test cannot break your recovery path. The full ticket this lab is based on is in [notes.md](notes.md) under **Story for today**.


## Setup — load the lab session

The cell below loads your lab variables into this CMD process. Today it sets more than usual, because you work with two authentication paths:

| Variable | What it is |
|----------|-----------|
| `%TOPIC%` and `%GROUP%` | Your own orders topic and consumer group |
| `%ACL_TOPIC%` | A separate throwaway topic used for the ACL exercise, so nothing you do there affects your orders topic |
| `%BOOTSTRAP%` and `%CLIENT%` | The SCRAM endpoint (port 9196) and its client properties file |
| `%BOOTSTRAP_IAM%` and `%CLIENT_IAM%` | The IAM endpoint (port 9198) and its client properties file |
| `%LOGIN%` | Your SCRAM username — the principal that ACLs are written against, as `User:%LOGIN%` |

**Before the IAM section (4b)**, two one-time preparation steps are needed:

1. Copy `samples/client-iam.properties.example` to `%USERPROFILE%\client-iam.properties`
2. Run `scripts\install-iam-jar.bat` once, to put the IAM authentication JAR on disk

**A note on the `log4j WARN` lines:** every Kafka command on Windows prints them. They mean the CLI has no logging configuration file, nothing more. They are not errors and they are not a sign that anything failed.


In [1]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo LOGIN=%LOGIN% TOPIC=%TOPIC% GROUP=%GROUP% ACL_TOPIC=%ACL_TOPIC%
echo BOOTSTRAP=%BOOTSTRAP%
echo BOOTSTRAP_IAM=%BOOTSTRAP_IAM%

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo LOGIN=%LOGIN% TOPIC=%TOPIC% GROUP=%GROUP% ACL_TOPIC=%ACL_TOPIC%
LOGIN=user15 TOPIC=orders-user15 GROUP=cg-user15-support ACL_TOPIC=acl-lab-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo BOOTSTRAP=%BOOTSTRAP%
BOOTSTRAP=b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com:9196

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo BOOTSTRAP_IAM=%BOOTSTRAP_IAM%
BOOTSTRAP_IAM=b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com:9198

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

## 1 — Review topic configuration

### Concept: two commands, two different answers

New engineers routinely run one of these, see nothing useful, and conclude the setting does not exist. They are looking in the wrong place.

| Command | What it shows you |
|---------|-------------------|
| `kafka-topics --describe` | Partitions, leaders, ISR, and the **effective** configuration on the topic line — including values inherited from the broker defaults |
| `kafka-configs --describe` | Only the **dynamic overrides** set specifically on this topic. Frequently empty. |

**An empty `kafka-configs --describe` is a normal, healthy result.** It means nobody has overridden anything, so the topic is using the broke’ defaults. It does **not** mean the topic has no retention.

This matters on a real ticket. When somebody asks *"what is the retention on this topic?"*, an empty override list is not the answer. You have to check the override first, and fall back to the broker default if there is no override.

**What you should see below:** replication factor **3**, full **ISR** on all three partitions, and a dynamic configuration that is either empty or shows explicit overrides.


In [2]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Topic: orders-user15	TopicId: u5fx7o-2RfW2LvjgNRKbPw	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,message.format.version=3.0-IV1,unclean.leader.election.enable=true
	Topic: orders-user15	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [3]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Dynamic configs for topic orders-user15 are:

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Reading — retention, and what it means for recovery

**Retention** is how long Kafka keeps a message before deleting it, whether or not anyone has read it. Two settings control it, and whichever limit is reached first wins:

| Setting | Meaning | Typical default |
|---------|---------|-----------------|
| `retention.ms` | Delete messages older than this | 7 days (`604800000`) |
| `retention.bytes` | Once a partition exceeds this size, delete its oldest data | often `-1`, meaning no size limit |

Where to find the value:

1. Look at `kafka-configs --describe` for a `retention.ms` override on the topic.
2. If there is no override, the broker default applies. The `Configs:` field on the `kafka-topics --describe` topic line shows the effective settings.

### The rule that governs every recovery request

> **You can only replay messages that are still on disk.**

`--to-earliest` moves the consumer grou’ position to the oldest message **still retained**. It does not reach back into deleted data. Once retention has removed a message, no offset reset, no replay, and no configuration change will bring it back — the bytes are gone.

So when someone asks *"can we reprocess last mont’ orders?"*, the very first thing you check is retention. If the topic keeps 7 days of data, the honest answer is no, and the sooner you say so the sooner they can look for the data somewhere else — a database, an archive, or the source system.

**Answer these before you promise anyone a recovery:**

| Question | Where the answer comes from |
|----------|----------------------------|
| How old is the missing message? | The ticket, or the produce’ logs |
| What is the topi’ retention? | `kafka-configs --describe`, then the broker default |
| Is the message still within retention? | Compare the two |
| Where else does this data exist? | The application team |


## 2 — Reset consumer offsets

### Checklist — before you reset anything

A missing message is very often **not** a Kafka problem, and a reset is the wrong first move. Rule these out first:

| Possible cause | How to check it | If this is the cause |
|----------------|-----------------|----------------------|
| The message was **never sent** | Producer logs show no successful publish for that id | Nothing to recover. The problem is upstream. |
| The group has **already read past** it | `--describe`: CURRENT-OFFSET is beyond that message | It was processed. Look at what the consumer did with it. |
| It went to a **different partition** | The message key decides the partition | It is there, just not where someone looked. |
| **Retention** deleted it | The message is older than `retention.ms` | Replay cannot recover it. Say so early. |

Reset offsets only when the data genuinely still exists on the broker **and** the grou’ saved position is genuinely wrong.

### Concept: what a reset actually changes

A consumer grou’ committed offset is a **bookmark** — nothing more. It records how far the group has read on each partition.

A reset moves the bookmark. It does **not** move any messages, does not delete anything, and does not re-send anything. The replay only happens afterwards, when a consumer starts up, sees the moved bookmark, and reads forward from there.

That distinction is worth being precise about, because it explains why a reset is both safe on the cluster and dangerous downstream. The cluster is unaffected. The downstream systems receive every one of those messages again.

### Where you can reset to

The lab uses `--to-earliest`, but production requests are rarely "replay everything". These are the options, and the third one is what real tickets usually need:

| Option | Where it moves the bookmark | When you would use it |
|--------|------------------------------|------------------------|
| `--to-earliest` | The oldest message still retained | A full replay of everything on disk |
| `--to-latest` | The end of the log | Abandoning a backlog you have decided not to process |
| `--to-datetime 2026-09-01T09:00:00.000` | The first offset at or after a timestamp | **"Replay from when the outage started"** — the most common real request |
| `--to-offset 4520` | One exact offset | You know precisely where to restart |
| `--shift-by -100` | Back (or forward) by N messages | A small, surgical correction |
| `--by-duration PT1H` | Back by a duration | "Redo the last hour" |

`--to-datetime` is worth remembering. It maps a business statement — *"we lost data between 09:00 and 09:40"* — onto exactly the messages involved, instead of replaying days of history to recover forty minutes of it.

### Concept: dry-run, then execute

Never reset in a single step. Follow this order every time:

1. **Stop every consumer in `%GROUP%`.** A running consumer keeps committing its own position and will fight your reset. Kafka refuses to reset an active group, and if you get past that you end up with a result neither you nor the consumer intended.
2. **`--dry-run`** — prints the plan of partition and new offset. Nothing changes.
3. **`--execute`** — applies it, only once the dry-run matches what you intended.
4. **`--describe`** — confirm CURRENT-OFFSET actually moved.

**The dry-run is your approval gate.** It prints the exact partition-to-offset plan. Paste that into the ticket, get it agreed, and only then run `--execute`. If `--execute` produces different numbers from the dry-run, something changed underneath you — stop and re-check rather than continuing.


In [4]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo === BEFORE reset ===
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo === BEFORE reset ===
=== BEFORE reset ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          2               2               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [5]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --dry-run

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --dry-run


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



GROUP                          TOPIC                          PARTITION  NEW-OFFSET     cg-user15-support              orders-user15                  0          0              cg-user15-support              orders-user15                  1          0              cg-user15-support              orders-user15                  2          0              
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [6]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --execute

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --topic %TOPIC% --reset-offsets --to-earliest --execute


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



GROUP                          TOPIC                          PARTITION  NEW-OFFSET     cg-user15-support              orders-user15                  0          0              cg-user15-support              orders-user15                  1          0              cg-user15-support              orders-user15                  2          0              
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [7]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo === AFTER reset ===
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo === AFTER reset ===
=== AFTER reset ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          0               1               1               -               -               -
cg-user15-support orders-user15   1          0               5               5               -               -               -
cg-user15-support orders-user15   2          0               2               2               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Reading — after execute, before replay

**What you should see:** `CURRENT-OFFSET` at **0** on each partition, with **`LAG` greater than 0**.

Read that state carefully, because it is the whole point of the exercise:

| Column | Value | What it tells you |
|--------|-------|-------------------|
| `CURRENT-OFFSET` | `0` | The bookmark has been moved back to the start |
| `LOG-END-OFFSET` | unchanged | No messages were added or removed — the log is exactly as it was |
| `LAG` | now greater than 0 | Those messages are waiting to be read again |

For example `LAG` of 1, 5 and 2 across three partitions means eight retained messages are queued for redelivery.

**Nothing has been replayed yet.** The bookmark moved; no consumer has run. This is the last moment at which the reset is still harmless — once a consumer starts, those messages go downstream for real. In production this pause is where you would confirm the downstream systems are ready.


## 3 — Replay messages

### Concept: what replay actually is

Replay is not a special Kafka feature or command. It is just a normal consumer reading messages it has read before, because you moved its bookmark backwards.

The messages were never removed. Kafka keeps everything until retention expires, regardless of who has read it — which is exactly what makes replay possible at all.

**How this lab differs from a real server.** On a server you would leave `consume.bat` running and stop it with Ctrl+C when the output settles. A notebook cell cannot be interrupted like that, so the cell below uses `--max-messages` and `--timeout-ms` to stop on its own.

**What you should see:** messages from the earlier lab days — Day 1 `order-*` lines, the Day 2 `recover-probe`, the Day 3 load-test records — followed by `Processed a total of N messages`. Once the consumer has caught up, `--describe` shows `LAG` back at 0.


In [8]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\consume.bat --from-beginning --max-messages 25 --timeout-ms 30000

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\consume.bat --from-beginning --max-messages 25 --timeout-ms 30000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.consumer.ConsumerConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


order-1001
order-1002
recover-probe
day1-test-msg
order-notebook-test
recover-probe
recover-probe
recover-probe


Processed a total of 8 messages



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Reading — the replay output

**What you should see:** a long run of historical lines from Days 1 to 4, and the same message appearing more than once — several `recover-probe` lines, for example.

**Those duplicates are correct, not a bug.** You have deliberately asked the group to re-read messages it had already processed. Kafka delivered exactly what you asked for.

### Why this is the most important slide of the day

In this lab a duplicate prints a harmless line of text. In production, the same consumer might have:

| What the consumer does with a message | What a duplicate causes |
|---------------------------------------|--------------------------|
| Inserts a row into a database | A duplicate record, or a primary-key violation |
| Sends a confirmation email | The customer is emailed twice |
| Charges a card | **The customer is charged twice** |
| Ships an order | A second shipment leaves the warehouse |

Kafka guarantees the messages are delivered. It cannot know whether processing them twice is harmless or catastrophic — only the application team knows that.

**The engineering answer is idempotent processing:** the consumer checks whether it has already handled this message (usually by a business key such as order id) and skips it if so. A consumer built that way can be replayed freely. A consumer that was not built that way needs the precautions in the next section.

After the replay completes, `--describe` should show `LAG` back at 0.


### Reprocessing safely — the practices that matter

Replay is easy to run and hard to run safely. The command takes seconds; the consequences can last days. These five practices are what separate a controlled replay from an incident you created yourself.

**1. Get the owne’ agreement, in writing, on the plan**

Not "can we replay?" but "we will move group `X` on topic `Y` back to offset `Z`, which is roughly N messages, and here is the dry-run output". The owner of the consuming application is the only person who knows what a duplicate does in their system. Ticket comments count as writing; a corridor conversation does not.

**2. Establish what a duplicate actually costs**

Ask the application team one direct question: *"if your service receives these messages a second time, what happens?"* The answer decides everything that follows. "Nothing, we key on order id" leads to a very different plan from "we would re-charge the customer".

**3. Pause the producer, if duplicates are harmful**

**What this means.** Ask the application team to temporarily stop the service that publishes to the topic, for the few minutes the replay takes. Kafka has no switch for this — producers are separate applications, so it is a request you make, not a command you run.

**Why it helps.** During a replay the consumer is working through old messages. If new messages keep arriving at the same time, old and new work are mixed together, and you cannot tell which downstream effects came from your replay and which came from normal traffic. Pausing the producer gives you a quiet window where only the replayed messages are moving, so you can verify the outcome before live traffic resumes.

**When it is worth the disruption.** Only when a duplicate does real damage — a second payment, a second shipment, a duplicate customer email. If the consumer is idempotent and duplicates are harmless, stopping a production producer is unnecessary disruption, and asking for it will cost you credibility.

**The trade-off to state out loud.** While the producer is paused, new messages are not being published at all. That is usually acceptable for a few minutes, but it is a second outage on top of the first, and the owner needs to agree to it rather than discover it.

**4. Validate with evidence, not assumption**

Two pieces of evidence, both of which you already know how to collect:

- `--describe` shows `LAG` back at 0, so the replay finished rather than stalling halfway
- A test message published *after* the replay is consumed successfully, proving live traffic is flowing again — the same technique as Day 2 section 4

"The consumer is running" is not validation. It only tells you a process started.

**5. Put the dry-run output in the ticket**

The dry-run is the record of what you intended. Six months later, when someone asks why a batch of orders was processed twice on a particular afternoon, that output is the difference between a documented, approved operation and an unexplained anomaly.


In [9]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          2               2               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

## 4a — SCRAM authorization (ACLs)

### Concept: authentication and authorization are two separate gates

These get used interchangeably in conversation, and confusing them sends you looking in completely the wrong place.

| Gate | Question it answers | How it fails |
|------|---------------------|--------------|
| **Authentication** — SCRAM on port 9196 | *Who are you?* Proves you are `User:%LOGIN%` | Wrong username or password. You cannot connect at all. |
| **Authorization** — ACLs | *Now that I know who you are, are you allowed to do this?* | You connect fine, then a specific operation is refused. |

Authentication happens first. Authorization is checked afterwards, per operation, per resource.

### Reading the exception you get

The error name tells you which gate stopped you, and that determines who you escalate to:

| Exception | What it means | What fixes it |
|-----------|---------------|---------------|
| `SaslAuthenticationException` | Authentication failed — bad credentials | Correct the username or password |
| `TopicAuthorizationException` | Authenticated, but not allowed on **this topic** | An ACL change on that topic |
| `GroupAuthorizationException` | Authenticated, but not allowed to use **this consumer group** | An ACL change on that group |
| `ClusterAuthorizationException` | Authenticated, but not allowed to perform a **cluster-wide** operation such as editing ACLs | A broader permission, usually held by the platform team |

**`TopicAuthorizationException` on produce is a permissions problem, full stop.** No offset reset, consumer restart, or retention change will help. Someone has to grant the permission. Recognising this instantly saves hours of investigating the wrong layer.

### What you will do in this section

The exercise runs on **`%ACL_TOPIC%`**, a throwaway topic, so nothing here can affect your orders topic. Same flow as [commands.md](commands.md) §4a:

1. List the current ACLs  
2. Produce successfully (`acl-ok`)  
3. **WAIT** until the room announces **GO DENIED** (ops / platform Deny Write is in place)  
4. Produce again → expect `TopicAuthorizationException`  
5. **WAIT** until the room announces **GO RESTORED**  
6. Produce successfully again (`acl-restored`)

Watching produce work, then break, then work again — with only the permission changing — is what makes the error message stick.

AWS Console admin is **not** the same as SCRAM permission to edit Kafka ACLs. Support keeps the same client; ops changes the ACL.


In [10]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-acls.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list --topic %ACL_TOPIC%

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-acls.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list --topic %ACL_TOPIC%


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Current ACLs for resource `ResourcePattern(resourceType=TOPIC, name=acl-lab-user15, patternType=LITERAL)`: 
 	(principal=User:user15, host=*, operation=DESCRIBE_CONFIGS, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=DESCRIBE, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=READ, permissionType=ALLOW)
	(principal=User:user15, host=*, operation=WRITE, permissionType=ALLOW) 


c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [11]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat %ACL_TOPIC% acl-ok

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\produce.bat %ACL_TOPIC% acl-ok


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo WAIT until the room announces GO DENIED — then run the next produce cell.
echo Do not change offsets while waiting.


### Reading — the ACL deny exercise

| Step | Expected result | What it proves |
|------|-----------------|----------------|
| `kafka-acls --list` | Existing Allow rules on `%ACL_TOPIC%` | Starting permission state |
| Produce `acl-ok` | Succeeds | Authenticated **and** authorized |
| After **GO DENIED** | Produce → **`TopicAuthorizationException`** | Authz is per topic / operation |
| After **GO RESTORED** | Produce `acl-restored` succeeds | Only the ACL changed |

### Why the before-and-after pair matters

Produce succeeding, then failing, then succeeding again, with **only the ACL changing**, isolates the cause. That is the evidence you want on a real authorization ticket.

### The one sentence to remember

**`TopicAuthorizationException` means authentication succeeded.** Your credentials and network path work; only Write was refused. No offset reset will help — the Deny must be removed (or an Allow restored).


In [13]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo Expect TopicAuthorizationException:
call ..\scripts\produce.bat %ACL_TOPIC% acl-denied-test

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo Expect TopicAuthorizationException:
Expect TopicAuthorizationException:

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\produce.bat %ACL_TOPIC% acl-denied-test


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo WAIT until the room announces GO RESTORED — then run the next produce cell.


In [15]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat %ACL_TOPIC% acl-restored

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\produce.bat %ACL_TOPIC% acl-restored


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

## 4b — IAM listener (a second authentication path)

### Concept: one cluster, several front doors

An MSK cluster can accept connections on several ports at once, each with its own authentication method. The cluster and the data are the same; only the way you prove your identity differs.

| Port | Authentication | Client configuration | Who typically uses it |
|------|----------------|----------------------|------------------------|
| **9196** | SCRAM over TLS — username and password | `%CLIENT%` | Applications outside AWS, or anything that cannot assume an IAM role |
| **9198** | IAM over TLS — AWS identity | `%CLIENT_IAM%` plus the IAM JAR | Applications running inside AWS on EC2, ECS, Lambda |

### Why a second door gets added at all

Picture a cluster that started with SCRAM only. Every application gets a Kafka username and password, and that works well for a while. The problems appear as the number of applications grows.

| What happens | Why it becomes a problem |
|--------------|--------------------------|
| Every application carries a Kafka password | Rotating one means updating the secret **and** every client that uses it. Miss one and it fails hours later. |
| Kafka keeps its own list of users | Kafka knows `User:orders-svc`, AWS knows an IAM role. Nobody sees both lists in one place, so access reviews get done twice. |
| Applications on AWS already have an identity | A service on ECS, EKS or Lambda runs under an IAM role. A second Kafka password is one more secret to store and possibly leak. |

IAM authentication removes that extra password: the application connects with the **AWS identity it already has**, and permission comes from an **IAM policy** (actions such as `kafka-cluster:Connect` and `kafka-cluster:WriteData` on named topics) rather than from Kafka ACLs.

One practical consequence: on the IAM door, `kafka-acls --list` shows nothing useful, because no Kafka ACLs are involved. An IAM refusal appears in the **clien’ own log**, so on an IAM ticket you ask for the application log instead of a list of ACLs.

**Do you need both?** No. IAM on its own is enough when every producer and consumer can authenticate with IAM. SCRAM stays when some clients cannot get AWS credentials — applications outside AWS, partner integrations, third-party tools. Neither choice is wrong; the useful question on a ticket is which door this client was built for, and whether it is using the matching port and file.

### What a successful SCRAM `--list` on 9196 already proved

Earlier cells connected on 9196 with `%CLIENT%` and listed topics. That single success proved three things at once: TLS negotiated correctly, your SCRAM credentials were accepted, and the network path to the broker is open. There is no need for `openssl` or any separate certificate check in this lab.

### The deliberate failure in this section

One cell intentionally sends the **SCRAM** client file to the **IAM** port. It will fail, and the error is the lesson: a listener accepts only its own authentication mechanism. Getting `SCRAM-SHA-512 not enabled` or a mechanism list of `[OAUTHBEARER, AWS_MSK_IAM]` is the expected, correct result.

This is worth doing once deliberately, because in production it shows up as a confusing incident: an application that worked yesterday starts failing to authenticate, and the actual cause is a configuration change that pointed it at the wrong port.


### Before IAM `--list`

The IAM listener (port **9198**) needs three things in place first:

1. `%USERPROFILE%\client-iam.properties` — copy it from `samples/client-iam.properties.example`
2. Run `scripts\install-iam-jar.bat` **once** to place the IAM auth JAR on disk
3. Add that JAR to `CLASSPATH` in the same cell:

`set CLASSPATH=C:\kafka\kafka_2.13-3.8.1\libs\aws-msk-iam-auth.jar;%CLASSPATH%`

If IAM is set up correctly, the topic list looks the same as the SCRAM `--list` on 9196. If it is empty or errors, that is fine for today — SCRAM on 9196 already proves your connectivity.


In [1]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
set CLASSPATH=C:\kafka\kafka_2.13-3.8.1\libs\aws-msk-iam-auth.jar;%CLASSPATH%
kafka-topics.bat --bootstrap-server %BOOTSTRAP_IAM% --command-config %CLIENT_IAM% --list

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


acl-lab-user15
orders-demo
orders-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

In [17]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo Expect failure — SCRAM client on IAM port:
kafka-topics.bat --bootstrap-server %BOOTSTRAP_IAM% --command-config %CLIENT% --list

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>echo Expect failure — SCRAM client on IAM port:
Expect failure — SCRAM client on IAM port:

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>kafka-topics.bat --bootstrap-server %BOOTSTRAP_IAM% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Error while executing topic command : Client SASL mechanism 'SCRAM-SHA-512' not enabled in the server, enabled mechanisms are [OAUTHBEARER, AWS_MSK_IAM]

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-04>

### Reading — IAM listener

**What you should see (wrong mix on purpose):** an error like `SCRAM-SHA-512 not enabled` or mechanisms `[OAUTHBEARER, AWS_MSK_IAM]`. That happens because port **9198** expects **IAM**, not the SCRAM client file. This proves listener and auth type must match.

**IAM `--list` (prior cell):** needs `set CLASSPATH=...aws-msk-iam-auth.jar` and `%CLIENT_IAM%`. When configured, topic list matches SCRAM `--list` on 9196.

**If IAM `--list` fails or is empty:** check `%CLIENT_IAM%` and IAM policy. SCRAM on **9196** still validates the rest of the lab.


## Assignment

Fill in [samples/assignment-recovery.md](samples/assignment-recovery.md). Write it as though it were a change record that someone else will read later.

| Section | What to include |
|---------|-----------------|
| **Topic configuration** | Retention on `%TOPIC%` — the value, and whether it is a topic override or the broker default |
| **Dry-run output** | The partition-to-offset plan exactly as printed, before you executed anything |
| **State after execute** | `CURRENT-OFFSET` and `LAG` per partition, showing the bookmark moved but nothing was consumed yet |
| **Replay validation** | One historical message you saw again, plus `LAG` returning to 0 |
| **Duplicate risk** | What a duplicate would cost downstream, and whether you would have paused the producer |
| **Authorization** | The exception you got while the deny was active, and why an offset reset would not have fixed it |

**The judgement question, and the one that carries the most marks:** a colleague asks you to reset a production consumer group to `--to-earliest` because "a few orders are missing". Write the two or three questions you would ask before touching anything, and say which reset target you would probably recommend instead.
